In [1]:
import pandas as pd
import pickle

import ipywidgets as widgets
from ipywidgets import interact

from src.data.data_load import (
    load_tables, 
    load_online_instance, 
    load_distances, 
    upload_ONLINE_static_solution
)
from src.data.solution_load import load_solution_dfs
from src.utils.filtering import flexible_filter
from src.utils.plotting import plot_metrics_comparison_dynamic
from src.data.metrics import collect_results_to_df, compute_metrics_with_moves, get_day_plotting_df
from src.config.experimentation_config import *
from src.config.config import *

data_path = '../data'

instance = 'instAD3'

distance_type = 'osrm'              # Options: ['osrm', 'manhattan']
dist_method = 'haversine'      # Options: ['precalced', 'haversine']

optimization_obj = 'driver_distance'

directorio_df, labors_raw_df, cities_df, duraciones_df, valid_cities = load_tables(data_path, generate_labors=False)
labors_real_df, labors_static_df, labors_dynamic_df = load_online_instance(data_path, instance, labors_raw_df)
dist_dict = load_distances(data_path, distance_type, instance, dist_method)

fechas = fechas_map(instance)

metricas = ['service_count', 'vt_count', 'num_drivers', 'driver_extra_time', 'driver_move_distance']

# Cargar resultados

In [2]:
algo_selection = {
    'Histórico': True,
    'OFFLINE': True,
    'INSERT': True,
    'BUFFER_FIXED': True,
    'REACT': True,
    'BUFFER_REACT': True,
    'ALFRED': True
}

In [3]:
algorithms = []
include_all_city = True

for algo, flag in algo_selection.items():
    if flag:
        labors_algo_df, moves_algo_df, postponed_labors = load_solution_dfs(
            data_path, 
            instance, 
            dist_method,
            algorithm=algo,
            include_all_city=include_all_city
        )
        algorithms.append({
            "name": algo,
            "labors_df": labors_algo_df,
            "moves_df": moves_algo_df,
            "type": "algorithm" if algo != 'Histórico' else 'historic',
            "color": algo_colors[algo],
            "visible": True,
        })

# Resultados

In [4]:
def plot_metrics_comparison_interactive_dynamic(
    algorithms, cities, metricas, dist_dict, fechas, metric_visualization_params=None
):
    """Fully dynamic interactive version for any number of algorithms."""

    city_dropdown = widgets.Dropdown(
        options=['ALL'] + sorted(list(set(cities))),
        value='ALL',
        description="Ciudad:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="300px")
    )

    # dynamically create visibility checkboxes
    algo_checks = {
        algo["name"]: widgets.Checkbox(value=algo.get("visible", True), description=algo["name"])
        for algo in algorithms
    }

    def _plot(city, **vis_flags):
        algo_copy = [{**algo, "visible": vis_flags[algo["name"]]} for algo in algorithms]
        plot_metrics_comparison_dynamic(
            algo_copy,
            city=city,
            metricas=metricas,
            dist_dict=dist_dict,
            fechas=fechas,
            metric_visualization_params=metric_visualization_params,
            save_dir=None
        )

    interact(
        _plot,
        city=city_dropdown,
        **algo_checks
    )

metrics_with_gap = [
    "total_distance",
    "driver_move_distance",
    "labor_extra_time",
    "driver_extra_time"
]

metric_visualization_params = {
    'service_count': (True, False), 
    'vt_count': (True, False), 
    'num_drivers': (False, True), 
    'driver_extra_time': (False, True), 
    'driver_move_distance': (False, True)
}

plot_metrics_comparison_interactive_dynamic(
    algorithms,
    cities=valid_cities,
    metricas=metricas,
    dist_dict=dist_dict,
    fechas=(fechas[0], fechas[-1]),
    metric_visualization_params=metric_visualization_params
)


interactive(children=(Dropdown(description='Ciudad:', layout=Layout(width='300px'), options=('ALL', '1', '1004…

# Validación de soluciones

In [5]:
from src.algorithms.solution_validator import validate_all_algorithms
validation_summary, moves_issues = validate_all_algorithms(
    algorithms[1:], 
    dist_method=dist_method,
    dist_dict=dist_dict
    )
display(validation_summary)



🚀 Validating OFFLINE ...
🧑‍🔧 Checked 40 drivers, found 0 overlaps
✅ Validated 227 drivers, 3003 rows, 0 inconsistencies

🚀 Validating INSERT ...
🧑‍🔧 Checked 42 drivers, found 109 overlaps
✅ Validated 239 drivers, 2994 rows, 10 inconsistencies

🚀 Validating BUFFER_FIXED ...
🧑‍🔧 Checked 42 drivers, found 145 overlaps
✅ Validated 233 drivers, 3012 rows, 16 inconsistencies

🚀 Validating REACT ...
🧑‍🔧 Checked 42 drivers, found 1 overlaps
✅ Validated 231 drivers, 2979 rows, 1 inconsistencies

🚀 Validating BUFFER_REACT ...
🧑‍🔧 Checked 42 drivers, found 1 overlaps
✅ Validated 235 drivers, 3000 rows, 1 inconsistencies

🚀 Validating ALFRED ...
🧑‍🔧 Checked 42 drivers, found 132 overlaps
✅ Validated 231 drivers, 2892 rows, 14 inconsistencies

📊 Validation Summary:


,algorithm,n_labors,n_moves,overlaps,move_inconsistencies,drivers_checked,rows_checked,pairs_checked,distance_calls,inconsistencies
0,OFFLINE,1180,3003,0,0,227,3003,3003,3003,0
1,INSERT,1160,2994,109,10,239,2994,2994,2994,10
2,BUFFER_FIXED,1139,3012,145,16,233,3012,3012,3012,16
3,REACT,1028,2979,1,1,231,2979,2979,2979,1
4,BUFFER_REACT,1037,3000,1,1,235,3000,3000,3000,1
5,ALFRED,1092,2892,132,14,231,2892,2892,2892,14


# SCHEDULING LOGIC IN DETAIL


In [6]:
import pandas as pd
import plotly.graph_objects as go


def plot_service_driver_distance_all_algorithms(
    labors_hist_df: pd.DataFrame,
    moves_hist_df: pd.DataFrame,
    algorithms: list,
    algorithms_info: list,
    city: str,
    date: str,
    top_n: int = 15,
    reference_algorithm: str = None
):
    """
    Plot driver move distance per service for ALL algorithms vs historic.

    algorithms: list of dicts
        [
            {"name": "REACT", "labors_df": ..., "moves_df": ...},
            ...
        ]
    """

    # --------------------------------------------------
    # Helper: compute distance per service
    # --------------------------------------------------
    def _compute_service_driver_distances(moves_df: pd.DataFrame) -> pd.DataFrame:
        df = moves_df[moves_df["labor_category"] == "DRIVER_MOVE"].copy()
        if df.empty:
            return pd.DataFrame(columns=["service_id", "driver_move_distance"])
        return (
            df.groupby("service_id", as_index=False)["distance_km"]
            .sum()
            .rename(columns={"distance_km": "driver_move_distance"})
        )

    # --------------------------------------------------
    # Filter HISTORIC
    # --------------------------------------------------
    moves_hist_filt = moves_hist_df[
        (moves_hist_df["city"] == city) &
        (moves_hist_df["schedule_date"].dt.strftime("%Y-%m-%d") == date)
    ].copy()

    moves_hist_filt["service_id"] = moves_hist_filt["service_id"].astype(str)
    hist_metrics = _compute_service_driver_distances(moves_hist_filt)

    # --------------------------------------------------
    # Filter ALGORITHMS
    # --------------------------------------------------
    algo_metrics_by_name = {}

    for algo in algorithms:
        name = algo["name"]
        moves_df = algo["moves_df"]

        moves_filt = moves_df[
            (moves_df["city"] == city) &
            (moves_df["schedule_date"].dt.strftime("%Y-%m-%d") == date)
        ].copy()

        moves_filt["service_id"] = moves_filt["service_id"].astype(str)
        algo_metrics_by_name[name] = _compute_service_driver_distances(moves_filt)

    # --------------------------------------------------
    # Select reference algorithm for top-N services
    # --------------------------------------------------
    if reference_algorithm is None:
        reference_algorithm = algorithms[0]["name"]

    ref_df = algo_metrics_by_name.get(reference_algorithm)
    if ref_df is None or ref_df.empty:
        raise ValueError("Reference algorithm has no data.")

    top_services = (
        ref_df.sort_values("driver_move_distance", ascending=False)
        .head(top_n)["service_id"]
        .tolist()
    )

    # --------------------------------------------------
    # Align all metrics to top services
    # --------------------------------------------------
    hist_metrics = hist_metrics[hist_metrics["service_id"].isin(top_services)]
    hist_metrics = hist_metrics.set_index("service_id").reindex(top_services).fillna(0)

    aligned_algo_metrics = {}
    for name, df in algo_metrics_by_name.items():
        aligned_algo_metrics[name] = (
            df[df["service_id"].isin(top_services)]
            .set_index("service_id")
            .reindex(top_services)
            .fillna(0)
        )

    # --------------------------------------------------
    # Colors
    # --------------------------------------------------
    algo_colors = {a["name"]: a.get("color", None) for a in algorithms_info}

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    fig = go.Figure()

    # Historic first
    fig.add_trace(go.Bar(
        x=top_services,
        y=hist_metrics["driver_move_distance"],
        name="Historic",
        marker_color="#7F7F7F"
    ))

    # Algorithms
    for algo in algorithms:
        name = algo["name"]
        df = aligned_algo_metrics[name]

        fig.add_trace(go.Bar(
            x=top_services,
            y=df["driver_move_distance"],
            name=name,
            marker_color=algo_colors.get(name)
        ))

    fig.update_layout(
        barmode="group",
        title=f"Driver Move Distance per Service — City {city}, Date {date}",
        xaxis_title="Service ID",
        yaxis_title="Driver Move Distance (km)",
        height=550,
        width=1100,
        legend=dict(
            orientation="h",
            y=-0.25,
            x=0.5,
            xanchor="center"
        ),
        margin=dict(t=60, b=80),
        plot_bgcolor="white"
    )

    fig.update_xaxes(tickangle=-45)
    fig.update_yaxes(showgrid=True, gridcolor="#E0E0E0")

    fig.show()


In [7]:
from ipywidgets import interact, Dropdown
import pandas as pd


def interactive_service_driver_distance_plot(
    algorithms: list,
    algorithms_info: list,
    top_n: int = 15
):
    """
    Interactive plot with City + Date dropdowns
    """

    labors_hist_df = algorithms[0]['labors_df']
    moves_hist_df = algorithms[0]['moves_df']

    # --------------------------------------------
    # Extract available cities & dates
    # --------------------------------------------
    cities = sorted(
        set(labors_hist_df["city"].dropna().astype(str).unique())
    )

    dates = sorted(
        labors_hist_df["schedule_date"]
        .dt.strftime("%Y-%m-%d")
        .unique()
        .tolist()
    )

    # --------------------------------------------
    # Widgets
    # --------------------------------------------
    city_dropdown = Dropdown(
        options=cities,
        value=cities[0],
        description="City:"
    )

    date_dropdown = Dropdown(
        options=dates,
        value=dates[0],
        description="Date:"
    )

    # --------------------------------------------
    # Update function
    # --------------------------------------------
    def _update(city: str, date: str):

        plot_service_driver_distance_all_algorithms(
            labors_hist_df=labors_hist_df,
            moves_hist_df=moves_hist_df,
            algorithms=algorithms[1:],
            algorithms_info=algorithms_info,
            city=city,
            date=date,
            top_n=top_n
        )

    # --------------------------------------------
    # Bind widgets
    # --------------------------------------------
    interact(
        _update,
        city=city_dropdown,
        date=date_dropdown
    )

interactive_service_driver_distance_plot(
    algorithms,
    algorithms
)

interactive(children=(Dropdown(description='City:', options=('1', '1004', '126', '149', '150', '830', '844', '…

In [8]:
# ============================================================
# Interactive Gantt Dashboard: City / Date / Algorithm
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

from src.utils.filtering import flexible_filter
from src.utils.plotting import (
    plot_gantt_by_services,
    plot_gantt_by_drivers
)
algo_by_name = {a["name"]: a for a in algorithms}

# ------------------------------------------------------------
# 2. Dynamic widget values (use any algorithm as reference)
# ------------------------------------------------------------
ref_labors_df = algorithms[0]["labors_df"]

cities = sorted(ref_labors_df["city"].dropna().unique())
algos = sorted(algo_by_name.keys())

# ------------------------------------------------------------
# 3. Widgets
# ------------------------------------------------------------
city_widget = widgets.Dropdown(
    options=['ALL'] + valid_cities,
    value='149',
    description="City:",
    layout=widgets.Layout(width="250px")
)

date_widget = widgets.Dropdown(
    options=fechas,
    value=fechas[0],
    description="Date:",
    layout=widgets.Layout(width="250px")
)

algo_widget = widgets.Dropdown(
    options=algos,
    value=algos[0],
    description="Algorithm:",
    layout=widgets.Layout(width="250px")
)

# ------------------------------------------------------------
# 4. Core update function
# ------------------------------------------------------------
def update_gantts(city, date, algorithm):
    clear_output(wait=True)

    algo = algo_by_name[algorithm]
    labors_df = algo["labors_df"]
    moves_df = algo["moves_df"]

    # Filter data
    labors_filt = flexible_filter(
        labors_df,
        city=city,
        schedule_date=date
    )

    moves_filt = flexible_filter(
        moves_df,
        city=city,
        schedule_date=date
    )

    if labors_filt.empty:
        print("No labors for the selected City / Date / Algorithm.")
        return
    
    if algorithm != 'Histórico':
        assignment_type="algorithm"
    else:
        assignment_type='historic'

    # --------------------------------------------------------
    # Plot 1: Gantt by services
    # --------------------------------------------------------
    plot_gantt_by_services(
        labors_filt,
        day_str=date,
        assignment_type=assignment_type,
        return_fig=False
    )

    # --------------------------------------------------------
    # Plot 2: Gantt by drivers
    # --------------------------------------------------------
    # if algorithm != 'Histórico':
    #     assignment_type = 'algorithm'
    # else:
    #     assignment_type = 'historic'
    plot_gantt_by_drivers(
        labors_filt,
        moves_filt,
        day_str=date,
        tiempo_gracia=TIEMPO_GRACIA,
        assignment_type=assignment_type,
        return_fig=False
    )

# ------------------------------------------------------------
# 5. Layout and display
# ------------------------------------------------------------
controls = widgets.HBox([
    city_widget,
    date_widget,
    algo_widget
])

out = widgets.interactive_output(
    update_gantts,
    {
        "city": city_widget,
        "date": date_widget,
        "algorithm": algo_widget
    }
)

display(controls, out)


Output()

# DEBUGGING

## Individual checks

In [10]:
city = '149'
fecha = '2026-01-05'

In [7]:
drop_columns = ['labor_type', 'labor_name', 'shop', 'alfred', 'labor_price', 
                'labor_created_at', 'labor_start_date', 'labor_end_date', 
                'client_type', 'paying_customer', 'state_service', 'labor_category',
                'start_address_id', 'end_address_id', 'date', 'address_id', 'address_point']
column_order = ['service_id', 'labor_id', 'created_at', 'schedule_date', 'actual_start', 'actual_end', 'map_start_point', 'map_end_point', 'city', 'assigned_driver', 'dist_km']

## NUMBER OF LABORS

In [12]:
filtered_labors_baseline = flexible_filter(algorithms[0]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_OFFLINE = flexible_filter(algorithms[1]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_INSERT = flexible_filter(algorithms[2]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_BUFFER_FIXED = flexible_filter(algorithms[3]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_REACT = flexible_filter(algorithms[4]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_REACT_BUFFER = flexible_filter(algorithms[5]['labors_df'],
                city=city,
                schedule_date=fecha)

filtered_labors_ALFRED = flexible_filter(algorithms[6]['labors_df'],
                city=city,
                schedule_date=fecha)


filtered_moves_baseline = flexible_filter(algorithms[0]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_OFFLINE = flexible_filter(algorithms[1]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_INSERT = flexible_filter(algorithms[2]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_BUFFER_FIXED = flexible_filter(algorithms[3]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_REACT = flexible_filter(algorithms[4]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_REACT_BUFFER = flexible_filter(algorithms[5]['moves_df'],
                city=city,
                schedule_date=fecha)

filtered_moves_ALFRED = flexible_filter(algorithms[6]['moves_df'],
                city=city,
                schedule_date=fecha)

print(f'-------- Labors')
print(f'Baseline:      {len(filtered_labors_baseline)}')
print(f'INSERT:        {len(filtered_labors_INSERT)}')
print(f'INSERT BUFFER: {len(filtered_labors_BUFFER_FIXED)}')
print(f'REACT:         {len(filtered_labors_REACT)}')
print(f'REACT BUFFER:  {len(filtered_labors_REACT_BUFFER)}')

print(f'\n-------- Moves')
print(f'Baseline:      {len(filtered_moves_baseline)}')
print(f'INSERT:        {len(filtered_moves_INSERT)}')
print(f'INSERT BUFFER: {len(filtered_moves_BUFFER_FIXED)}')
print(f'REACT:         {len(filtered_moves_REACT)}')
print(f'REACT BUFFER:  {len(filtered_moves_REACT_BUFFER)}')

-------- Labors
Baseline:      76
INSERT:        76
INSERT BUFFER: 76
REACT:         70
REACT BUFFER:  71

-------- Moves
Baseline:      213
INSERT:        213
INSERT BUFFER: 213
REACT:         210
REACT BUFFER:  213


## REACT doesn't having the 

In [ ]:
service_id = '256887'

In [14]:
flexible_filter(filtered_labors_baseline, service_id=service_id)


,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,address_name,labor_sequence,map_start_point,map_end_point,date,historic_driver,historic_start,historic_end,dist_km,actual_status
22,256887,353165,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,35000.0,2025-06-18 08:10:27.173000-05:00,2026-01-05 13:19:23-05:00,2026-01-05 14:21:26-05:00,10449,...,casa,0,POINT (-74.092767 4.5863035),POINT (-74.1257024196434 4.568297647307015),2026-01-05,10449,2026-01-05 12:30:00-05:00,2026-01-05 13:21:14.721263-05:00,4.163570,COMPLETED
23,256887,353166,3.0,Wash and Polish,WASH_AND_POLISH,33915.0,2025-06-18 08:10:27.250000-05:00,2026-01-05 14:21:26-05:00,2026-01-05 15:28:57-05:00,,...,Multiservicios Megamotors,1,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1257024196434 4.568297647307015),2026-01-05,<NA>,2026-01-05 13:21:14.721263-05:00,2026-01-05 14:37:44.221263-05:00,0.000000,COMPLETED
24,256887,353167,2.0,Alfred Transport,VEHICLE_TRANSPORTATION,35000.0,2025-06-18 08:10:27.315000-05:00,2026-01-05 15:28:57-05:00,2026-01-05 16:14:38-05:00,10449,...,casa,2,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1691584 4.6514129),2026-01-05,10449,2026-01-05 14:37:44.221263-05:00,2026-01-05 15:38:22.177917-05:00,10.421741,COMPLETED


In [15]:
flexible_filter(filtered_moves_baseline, service_id=service_id)


,service_id,labor_id,labor_context_id,labor_name,labor_category,historic_driver,schedule_date,historic_start,historic_end,start_point,end_point,distance_km,city,duration_min,date
132,256887,353165,353165_free,FREE_TIME,FREE_TIME,10449,2026-01-05 13:00:00-05:00,2026-01-05 10:24:51.934804-05:00,2026-01-05 12:11:41.594263-05:00,POINT (-74.05261759999999 4.658239599999999),POINT (-74.05261759999999 4.658239599999999),0.000000,149,106.8,2026-01-05
133,256887,353165,353165_move,DRIVER_MOVE,DRIVER_MOVE,10449,2026-01-05 13:00:00-05:00,2026-01-05 12:11:41.594263-05:00,2026-01-05 12:30:00-05:00,POINT (-74.05261759999999 4.658239599999999),POINT (-74.092767 4.5863035),9.153381,149,18.3,2026-01-05
134,256887,353165,353165_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,10449,2026-01-05 13:00:00-05:00,2026-01-05 12:30:00-05:00,2026-01-05 13:21:14.721263-05:00,POINT (-74.092767 4.5863035),POINT (-74.1257024196434 4.568297647307015),4.163570,149,51.2,2026-01-05
135,256887,353167,353167_free,FREE_TIME,FREE_TIME,10449,2026-01-05 13:00:00-05:00,2026-01-05 13:21:14.721263-05:00,2026-01-05 14:37:44.221263-05:00,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1257024196434 4.568297647307015),0.000000,149,76.5,2026-01-05
136,256887,353167,353167_move,DRIVER_MOVE,DRIVER_MOVE,10449,2026-01-05 13:00:00-05:00,2026-01-05 14:37:44.221263-05:00,2026-01-05 14:37:44.221263-05:00,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1257024196434 4.568297647307015),0.000000,149,0.0,2026-01-05
137,256887,353167,353167_labor,Alfred Transport,VEHICLE_TRANSPORTATION,10449,2026-01-05 13:00:00-05:00,2026-01-05 14:37:44.221263-05:00,2026-01-05 15:38:22.177917-05:00,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1691584 4.6514129),10.421741,149,60.6,2026-01-05


In [13]:
flexible_filter(filtered_labors_REACT, service_id=service_id)


,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,map_end_point,date,assigned_driver,actual_start,actual_end,dist_km,actual_status,old_assigned_driver,old_actual_start,old_actual_end
49,256887,353165,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,35000.0,2025-06-18 08:10:27.173000-05:00,2026-01-05 13:19:23-05:00,2026-01-05 14:21:26-05:00,10449,...,POINT (-74.1257024196434 4.568297647307015),2026-01-05,69861,2026-01-05 12:30:00-05:00,2026-01-05 13:21:14.721263-05:00,4.163570,COMPLETED,70934,2026-01-05 12:30:00-05:00,2026-01-05 13:21:14.721263-05:00
58,256887,353167,2.0,Alfred Transport,VEHICLE_TRANSPORTATION,35000.0,2025-06-18 08:10:27.315000-05:00,2026-01-05 15:28:57-05:00,2026-01-05 16:14:38-05:00,10449,...,POINT (-74.1691584 4.6514129),2026-01-05,70431,2026-01-05 12:35:49.239442-05:00,2026-01-05 13:36:27.196096-05:00,10.421741,COMPLETED,70934,2026-01-05 14:37:44.221263-05:00,2026-01-05 15:38:22.177917-05:00


In [16]:
flexible_filter(filtered_moves_REACT, service_id=service_id)


,service_id,labor_id,labor_context_id,labor_name,labor_category,assigned_driver,schedule_date,actual_start,actual_end,start_point,end_point,distance_km,city,duration_min,date
147,256887,353165,353165_free,FREE_TIME,FREE_TIME,69861,2026-01-05 13:00:00-05:00,2026-01-05 12:15:37.821000-05:00,2026-01-05 12:17:03.276094-05:00,POINT (-74.1301733 4.6310053),POINT (-74.1301733 4.6310053),0.000000,149,1.4,2026-01-05
148,256887,353165,353165_move,DRIVER_MOVE,DRIVER_MOVE,69861,2026-01-05 13:00:00-05:00,2026-01-05 12:17:03.276094-05:00,2026-01-05 12:30:00-05:00,POINT (-74.1301733 4.6310053),POINT (-74.092767 4.5863035),6.472699,149,12.9,2026-01-05
149,256887,353165,353165_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,69861,2026-01-05 13:00:00-05:00,2026-01-05 12:30:00-05:00,2026-01-05 13:21:14.721263-05:00,POINT (-74.092767 4.5863035),POINT (-74.1257024196434 4.568297647307015),4.163570,149,51.2,2026-01-05
174,256887,353167,353167_free,FREE_TIME,FREE_TIME,70431,2026-01-05 13:00:00-05:00,2026-01-05 12:19:21.571000-05:00,2026-01-05 12:19:21.571000-05:00,POINT (-74.1988555 4.5810069),POINT (-74.1988555 4.5810069),0.000000,149,0.0,2026-01-05
175,256887,353167,353167_move,DRIVER_MOVE,DRIVER_MOVE,70431,2026-01-05 13:00:00-05:00,2026-01-05 12:19:21.571000-05:00,2026-01-05 12:35:49.239442-05:00,POINT (-74.1988555 4.5810069),POINT (-74.1257024196434 4.568297647307015),8.230570,149,16.5,2026-01-05
176,256887,353167,353167_labor,Alfred Transport,VEHICLE_TRANSPORTATION,70431,2026-01-05 13:00:00-05:00,2026-01-05 12:35:49.239442-05:00,2026-01-05 13:36:27.196096-05:00,POINT (-74.1257024196434 4.568297647307015),POINT (-74.1691584 4.6514129),10.421741,149,60.6,2026-01-05


## EXTRA SERVICES

Apparently there are some services which are repeated (someway). This service should be cancelled completely if one of the services cannot be fulfilled.
Esto cause que INSERT E INSERTBUFFER tengan un aparente servicio adicional en 07.01.26.

In [9]:
filtered_labors_baseline

,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,address_point,address_name,labor_sequence,date,map_start_point,map_end_point,assigned_driver,actual_start,actual_end,dist_km
0,213400,306065,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,65004.0,2025-02-17 14:23:24.316000-05:00,2026-01-07 14:49:00-05:00,2026-01-07 15:35:00-05:00,70080,...,POINT (-75.6678067 4.799966),Casa,0,2026-01-07,POINT (-75.7362559 4.815196300000001),POINT (-75.7016694 4.811435),70080,2026-01-07 15:00:00-05:00,2026-01-07 15:50:46.953170-05:00,3.855035
1,213411,306080,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,65004.0,2025-02-17 14:42:00.422000-05:00,2026-01-07 16:39:00-05:00,2026-01-07 17:04:00-05:00,70080,...,POINT (-75.6678067 4.799966),Casa,0,2026-01-07,POINT (-75.7016694 4.811435),POINT (-75.7362559 4.815196300000001),70080,2026-01-07 16:30:00-05:00,2026-01-07 17:20:46.953170-05:00,3.855035


In [19]:
labors_algo_baseline_df.columns

Index(['service_id', 'labor_id', 'labor_type', 'labor_name', 'labor_category',
       'labor_price', 'labor_created_at', 'labor_start_date', 'labor_end_date',
       'alfred', 'shop', 'created_at', 'schedule_date', 'client_type',
       'paying_customer', 'state_service', 'start_address_id',
       'start_address_point', 'end_address_id', 'end_address_point', 'city',
       'address_id', 'address_point', 'address_name', 'labor_sequence', 'date',
       'map_start_point', 'map_end_point', 'assigned_driver', 'actual_start',
       'actual_end', 'dist_km'],
      dtype='object')

In [11]:
flexible_filter(
    labors_algo_baseline_df,
    service_id='232102'
).sort_values(['schedule_date', 'actual_start', 'actual_end'])

,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,address_point,address_name,labor_sequence,date,map_start_point,map_end_point,assigned_driver,actual_start,actual_end,dist_km
2,232102,327400,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-21 07:40:54.844000-05:00,2026-01-06 09:35:00-05:00,2026-01-06 11:38:00-05:00,72164,...,POINT (-75.7102592 4.801893499999999),Casa,4,2026-01-06,POINT (-75.7102592 4.801893499999999),POINT (-75.7102592 4.801893499999999),72164,2026-01-06 16:00:00-05:00,2026-01-06 16:45:00-05:00,0.000000
3,232102,327417,16.0,Tanqueo Terpel (Corriente),OTHER_LABORS,47600.0,2025-04-21 08:16:06.653000-05:00,2026-01-06 11:38:00-05:00,2026-01-06 12:00:00-05:00,,...,POINT (-75.6927012 4.810058499999999),Terpel - Pereira,6,2026-01-06,POINT (-75.6927012 4.810058499999999),POINT (-75.6927012 4.810058499999999),<NA>,2026-01-06 16:45:00-05:00,2026-01-06 16:49:11.178571-05:00,0.000000
4,232102,326475,2.0,Alfred Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-15 14:35:15.906000-05:00,2026-01-06 07:41:00-05:00,2026-01-06 10:03:00-05:00,72164,...,POINT (-75.7102592 4.801893499999999),Casa,2,2026-01-06,POINT (-75.7193049 4.8156245),POINT (-75.7102592 4.801893499999999),72164,2026-01-06 16:49:11.178571-05:00,2026-01-06 17:36:55.555335-05:00,1.826408
5,232102,327399,6.0,General Mechanics,GENERAL_MECHANICS,1242360.0,2025-04-21 07:40:54.796000-05:00,2026-01-06 10:03:00-05:00,2026-01-06 09:35:00-05:00,,...,POINT (-75.6770009 4.830667699999999),Serviautos Pereira,3,2026-01-06,POINT (-75.6770009 4.830667699999999),POINT (-75.6770009 4.830667699999999),<NA>,2026-01-06 17:36:55.555335-05:00,2026-01-07 09:02:03.412478-05:00,0.000000
10,232102,326473,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-15 14:32:57.273000-05:00,2026-01-06 16:00:00-05:00,2026-01-06 18:21:00-05:00,72164,...,POINT (-75.7102592 4.801893499999999),Casa,0,2026-01-06,POINT (-75.7102592 4.801893499999999),POINT (-75.7102592 4.801893499999999),<NA>,NaT,NaT,0.000000
9,232102,326474,4.0,Tecnomecanica,SAFETI_INSPECTION,318200.0,2025-04-15 14:32:57.325000-05:00,2026-01-06 18:21:00-05:00,2026-01-06 07:41:00-05:00,,...,POINT (-75.6613069 4.850945),CDA Cardisel SAS La Romelia - Pereira,1,2026-01-06,POINT (-75.6613069 4.850945),POINT (-75.6613069 4.850945),<NA>,NaT,NaT,0.000000
7,232102,327416,4.0,Tecnomecanica,SAFETI_INSPECTION,296128.0,2025-04-21 08:13:20.415000-05:00,2026-01-06 11:38:00-05:00,2026-01-06 14:18:00-05:00,,...,POINT (-75.6613069 4.850945),CDA Cardisel SAS La Romelia - Pereira,7,2026-01-06,POINT (-75.6613069 4.850945),POINT (-75.6613069 4.850945),<NA>,NaT,NaT,0.000000
6,232102,327418,755.0,Alfred Service Transport,VEHICLE_TRANSPORTATION,45000.0,2025-04-21 08:16:06.670000-05:00,2026-01-06 11:04:00-05:00,2026-01-06 11:38:00-05:00,72164,...,POINT (-75.7102592 4.801893499999999),Casa,5,2026-01-06,POINT (-75.6770009 4.830667699999999),POINT (-75.7102592 4.801893499999999),<NA>,NaT,NaT,0.000000
8,232102,327541,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-21 12:09:40.098000-05:00,2026-01-06 14:18:00-05:00,2026-01-06 14:32:00-05:00,72164,...,POINT (-75.7102592 4.801893499999999),Casa,8,2026-01-06,POINT (-75.6927012 4.810058499999999),POINT (-75.7102592 4.801893499999999),<NA>,NaT,NaT,0.000000


In [18]:
filtered_labors_INSERT

,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,labor_sequence,date,map_start_point,map_end_point,assigned_driver,actual_start,actual_end,dist_km,latest_arrival_time,n_drivers
1070,213400,306065,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,65004.0,2025-02-17 14:23:24.316000-05:00,2026-01-07 14:49:00-05:00,2026-01-07 15:35:00-05:00,70080,...,0,2026-01-07,NaN,NaN,70080,2026-01-07 15:00:00-05:00,2026-01-07 15:50:46.953170-05:00,3.855035,2026-01-07 15:45:00-05:00,NaN
1071,213411,306080,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,65004.0,2025-02-17 14:42:00.422000-05:00,2026-01-07 16:39:00-05:00,2026-01-07 17:04:00-05:00,70080,...,0,2026-01-07,NaN,NaN,70080,2026-01-07 16:30:00-05:00,2026-01-07 17:20:46.953170-05:00,3.855035,2026-01-07 17:15:00-05:00,NaN


In [13]:
flexible_filter(
    labors_algo_INSERT_df,
    service_id='232102'
).sort_values(['schedule_date', 'actual_start', 'actual_end'])

,service_id,labor_id,labor_type,labor_name,labor_category,labor_price,labor_created_at,labor_start_date,labor_end_date,alfred,...,labor_sequence,date,map_start_point,map_end_point,assigned_driver,actual_start,actual_end,dist_km,latest_arrival_time,n_drivers
1064,232102,326473,12.0,Alfred Initial Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-15 14:32:57.273000-05:00,2026-01-06 16:00:00-05:00,2026-01-06 18:21:00-05:00,72164,...,0,2026-01-06,NaN,NaN,72164,2026-01-06 16:00:00-05:00,2026-01-06 16:45:00-05:00,0.0,2026-01-06 16:45:00-05:00,NaN
1066,232102,326475,2.0,Alfred Transport,VEHICLE_TRANSPORTATION,57789.0,2025-04-15 14:35:15.906000-05:00,2026-01-06 07:41:00-05:00,2026-01-06 10:03:00-05:00,72164,...,2,2026-01-06,NaN,NaN,72164,2026-01-06 16:45:00-05:00,2026-01-06 17:30:00-05:00,0.0,2026-01-06 16:45:00-05:00,NaN
1065,232102,326474,4.0,Tecnomecanica,SAFETI_INSPECTION,318200.0,2025-04-15 14:32:57.325000-05:00,2026-01-06 18:21:00-05:00,2026-01-06 07:41:00-05:00,,...,1,2026-01-06,NaN,NaN,,2026-01-06 16:45:00-05:00,2026-01-06 18:33:30.500000-05:00,0.0,2026-01-06 16:45:00-05:00,NaN
1067,232102,327399,6.0,General Mechanics,GENERAL_MECHANICS,1242360.0,2025-04-21 07:40:54.796000-05:00,2026-01-06 10:03:00-05:00,2026-01-06 09:35:00-05:00,,...,3,2026-01-06,NaN,NaN,,2026-01-06 17:30:00-05:00,2026-01-07 12:53:15-05:00,0.0,2026-01-06 16:45:00-05:00,NaN
1068,232102,327417,16.0,Tanqueo Terpel (Corriente),OTHER_LABORS,47600.0,2025-04-21 08:16:06.653000-05:00,2026-01-06 11:38:00-05:00,2026-01-06 12:00:00-05:00,,...,6,2026-01-06,NaN,NaN,,2026-01-07 12:53:15-05:00,2026-01-07 13:00:42.750000-05:00,0.0,2026-01-06 16:45:00-05:00,NaN
1069,232102,327416,4.0,Tecnomecanica,SAFETI_INSPECTION,296128.0,2025-04-21 08:13:20.415000-05:00,2026-01-06 11:38:00-05:00,2026-01-06 14:18:00-05:00,,...,7,2026-01-06,NaN,NaN,,2026-01-07 13:00:42.750000-05:00,2026-01-07 14:49:13.250000-05:00,0.0,2026-01-06 16:45:00-05:00,NaN


In [12]:
plot_metrics_comparison_dynamic(
    algorithms,
    city=city,
    metricas=metricas,
    dist_dict=dist_dict,
    fechas=fechas,
    metrics_with_gap=metrics_with_gap
)



['274945' '275107' '275163']
['231146' '232092' '232102']
['213400' '213411']
['216642' '216714' '216754']
['211033' '211034' '211143']
['213675' '213732']
['226543' '226554' '226656']


['274945' '275107' '275163']
['231146' '232092' '232102']
['213400' '213411']
['216642' '216714' '216754']
['211033' '211034' '211143']
['213675' '213732']
['226543' '226554' '226656']


['274945' '275107' '275163']
['231146' '232092' '232102']
['232102' '213400' '213411']
['216642' '216714' '216754']
['211033' '211034' '211143']
['213675' '213732']
['226543' '226554' '226656']


['274945' '275107' '275163']
['231146' '232092' '232102']
['232102' '213400' '213411']
['216642' '216714' '216754']
['211033' '211034' '211143']
['213675' '213732']
['226543' '226554' '226656']


['274945' '275107' '275163']
['231146' '232092' '232102']
['213400' '213411']
['216642' '216714' '216754']
['211033' '211034' '211143']
['213675' '213732']
['226543' '226554' '226656']


['274945' '275107' '275163']
['231146' '23209

## OTHER TESTING!!!

In [19]:
flexible_filter(
    labors_algo_baseline_df.drop(
        columns=drop_columns
    )[column_order],
    city=city, 
    schedule_date=fecha).sort_values(['schedule_date'])

,service_id,labor_id,created_at,schedule_date,actual_start,actual_end,map_start_point,map_end_point,city,dist_km
0,218033,311188,2026-01-03 08:22:06.741000-05:00,2026-01-05 07:00:00-05:00,2026-01-05 06:30:00-05:00,2026-01-05 07:17:09.137485-05:00,POINT (-73.11255469999999 7.117752199999999),POINT (-73.1192449 7.1288176),844,1.434861
1,218109,311287,2026-01-03 12:01:47.748000-05:00,2026-01-05 09:30:00-05:00,2026-01-05 09:00:00-05:00,2026-01-05 09:59:36.286581-05:00,POINT (-73.0782802 7.0281457),POINT (-73.11350980000002 7.108425699999999),844,9.736518
2,218905,312140,2026-01-05 11:06:20.985000-05:00,2026-01-05 16:00:00-05:00,2026-01-05 15:30:00-05:00,2026-01-05 16:22:59.420799-05:00,POINT (-73.0975205 7.0649086),POINT (-73.1149525 7.109582199999998),844,5.326898
3,218944,312188,2026-01-05 13:44:35.523000-05:00,2026-01-05 18:00:00-05:00,2026-01-05 17:30:00-05:00,2026-01-05 18:29:36.286581-05:00,POINT (-73.11350980000002 7.108425699999999),POINT (-73.0782802 7.0281457),844,9.736518


In [18]:
flexible_filter(
    labors_algo_INSERT_BUFFER_df.drop(
        columns=drop_columns
    )[column_order], 
    city=city, 
    schedule_date=fecha).sort_values(['schedule_date'])

,service_id,labor_id,created_at,schedule_date,actual_start,actual_end,map_start_point,map_end_point,city,dist_km
0,218033,311188,2026-01-03 08:22:06.741000-05:00,2026-01-05 07:00:00-05:00,2026-01-05 06:30:00-05:00,2026-01-05 07:17:09.137485-05:00,POINT (-73.11255469999999 7.117752199999999),POINT (-73.1192449 7.1288176),844,1.434861
1,218109,311287,2026-01-03 12:01:47.748000-05:00,2026-01-05 09:30:00-05:00,2026-01-05 09:00:00-05:00,2026-01-05 09:59:36.286581-05:00,POINT (-73.0782802 7.0281457),POINT (-73.11350980000002 7.108425699999999),844,9.736518


In [20]:
flexible_filter(
    labors_algo_REACT_df.drop(
        columns=drop_columns
    )[column_order], 
    city=city, 
    schedule_date=fecha).sort_values(['schedule_date'])

,service_id,labor_id,created_at,schedule_date,actual_start,actual_end,map_start_point,map_end_point,city,dist_km
0,218033,311188,2026-01-03 08:22:06.741000-05:00,2026-01-05 07:00:00-05:00,2026-01-05 06:30:00-05:00,2026-01-05 07:17:09.137485-05:00,POINT (-73.11255469999999 7.117752199999999),POINT (-73.1192449 7.1288176),844,1.434861
1,218109,311287,2026-01-03 12:01:47.748000-05:00,2026-01-05 09:30:00-05:00,2026-01-05 09:00:00-05:00,2026-01-05 09:59:36.286581-05:00,POINT (-73.0782802 7.0281457),POINT (-73.11350980000002 7.108425699999999),844,9.736518
2,218905,312140,2026-01-05 11:06:20.985000-05:00,2026-01-05 16:00:00-05:00,2026-01-05 15:30:00-05:00,2026-01-05 16:22:59.420799-05:00,POINT (-73.0975205 7.0649086),POINT (-73.1149525 7.109582199999998),844,5.326898
3,218944,312188,2026-01-05 13:44:35.523000-05:00,2026-01-05 18:00:00-05:00,2026-01-05 17:30:00-05:00,2026-01-05 18:29:36.286581-05:00,POINT (-73.11350980000002 7.108425699999999),POINT (-73.0782802 7.0281457),844,9.736518


### Moves

In [24]:
flexible_filter(
    moves_algo_baseline_df, 
    city=city, 
    schedule_date=fecha).sort_values(['schedule_date'])


,service_id,labor_id,labor_context_id,labor_name,labor_category,assigned_driver,schedule_date,actual_start,actual_end,start_point,end_point,distance_km,duration_min,city,date
0,218033,311188,311188_free,FREE_TIME,FREE_TIME,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:23:45.091679-05:00,2026-01-05 06:23:45.091679-05:00,POINT (-73.1037471 7.0910492),POINT (-73.1037471 7.0910492),0.000000,0.0,844,2026-01-05
1,218033,311188,311188_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:23:45.091679-05:00,2026-01-05 06:30:00-05:00,POINT (-73.1037471 7.0910492),POINT (-73.11255469999999 7.117752199999999),3.124236,6.2,844,2026-01-05
2,218033,311188,311188_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:30:00-05:00,2026-01-05 07:17:09.137485-05:00,POINT (-73.11255469999999 7.117752199999999),POINT (-73.1192449 7.1288176),NaN,47.2,844,2026-01-05
3,218109,311287,311287_free,FREE_TIME,FREE_TIME,6412,2026-01-05 09:30:00-05:00,2026-01-05 07:17:09.137485-05:00,2026-01-05 08:35:51.307394-05:00,POINT (-73.1192449 7.1288176),POINT (-73.1192449 7.1288176),0.000000,78.7,844,2026-01-05
4,218109,311287,311287_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 09:30:00-05:00,2026-01-05 08:35:51.307394-05:00,2026-01-05 09:00:00-05:00,POINT (-73.1192449 7.1288176),POINT (-73.0782802 7.0281457),12.072438,24.1,844,2026-01-05
5,218109,311287,311287_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 09:30:00-05:00,2026-01-05 09:00:00-05:00,2026-01-05 09:59:36.286581-05:00,POINT (-73.0782802 7.0281457),POINT (-73.11350980000002 7.108425699999999),NaN,59.6,844,2026-01-05
6,218905,312140,312140_free,FREE_TIME,FREE_TIME,6412,2026-01-05 16:00:00-05:00,2026-01-05 09:59:36.286581-05:00,2026-01-05 15:19:41.939577-05:00,POINT (-73.11350980000002 7.108425699999999),POINT (-73.11350980000002 7.108425699999999),0.000000,320.1,844,2026-01-05
7,218905,312140,312140_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 16:00:00-05:00,2026-01-05 15:19:41.939577-05:00,2026-01-05 15:30:00-05:00,POINT (-73.11350980000002 7.108425699999999),POINT (-73.0975205 7.0649086),5.150504,10.3,844,2026-01-05
8,218905,312140,312140_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 16:00:00-05:00,2026-01-05 15:30:00-05:00,2026-01-05 16:22:59.420799-05:00,POINT (-73.0975205 7.0649086),POINT (-73.1149525 7.109582199999998),NaN,53.0,844,2026-01-05
9,218944,312188,312188_free,FREE_TIME,FREE_TIME,6412,2026-01-05 18:00:00-05:00,2026-01-05 16:22:59.420799-05:00,2026-01-05 17:29:35.443094-05:00,POINT (-73.1149525 7.109582199999998),POINT (-73.1149525 7.109582199999998),0.000000,66.6,844,2026-01-05


In [23]:
flexible_filter(
    moves_algo_REACT_df, 
    city=city, 
    schedule_date=fecha).sort_values(['schedule_date'])

,service_id,labor_id,labor_context_id,labor_name,labor_category,assigned_driver,schedule_date,actual_start,actual_end,start_point,end_point,distance_km,duration_min,city,date
0,218033,311188,311188_free,FREE_TIME,FREE_TIME,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:23:45.091679-05:00,2026-01-05 06:23:45.091679-05:00,POINT (-73.1037471 7.0910492),POINT (-73.1037471 7.0910492),0.000000,0.0,844,2026-01-05
1,218033,311188,311188_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:23:45.091679-05:00,2026-01-05 06:30:00-05:00,POINT (-73.1037471 7.0910492),POINT (-73.11255469999999 7.117752199999999),3.124236,6.2,844,2026-01-05
2,218033,311188,311188_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 07:00:00-05:00,2026-01-05 06:30:00-05:00,2026-01-05 07:17:09.137485-05:00,POINT (-73.11255469999999 7.117752199999999),POINT (-73.1192449 7.1288176),NaN,47.2,844,2026-01-05
3,218109,311287,311287_free,FREE_TIME,FREE_TIME,6412,2026-01-05 09:30:00-05:00,2026-01-05 07:17:09.137485-05:00,2026-01-05 08:35:51.307394-05:00,POINT (-73.1192449 7.1288176),POINT (-73.1192449 7.1288176),0.000000,78.7,844,2026-01-05
4,218109,311287,311287_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 09:30:00-05:00,2026-01-05 08:35:51.307394-05:00,2026-01-05 09:00:00-05:00,POINT (-73.1192449 7.1288176),POINT (-73.0782802 7.0281457),12.072438,24.1,844,2026-01-05
5,218109,311287,311287_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 09:30:00-05:00,2026-01-05 09:00:00-05:00,2026-01-05 09:59:36.286581-05:00,POINT (-73.0782802 7.0281457),POINT (-73.11350980000002 7.108425699999999),NaN,59.6,844,2026-01-05


In [31]:
df = moves_algo_baseline_df
df = df[df['city'] != 'ALL']
df

,service_id,labor_id,labor_context_id,labor_name,labor_category,assigned_driver,schedule_date,actual_start,actual_end,start_point,end_point,distance_km,duration_min,city,date
0,205621,297525,297525_free,FREE_TIME,FREE_TIME,10491,2026-01-05 07:30:00-05:00,2026-01-05 06:53:04.404846-05:00,2026-01-05 06:53:04.404846-05:00,POINT (-75.5948708 6.222133500000001),POINT (-75.5948708 6.222133500000001),0.000000,0.0,1,2026-01-05
1,205621,297525,297525_move,DRIVER_MOVE,DRIVER_MOVE,10491,2026-01-05 07:30:00-05:00,2026-01-05 06:53:04.404846-05:00,2026-01-05 07:00:00-05:00,POINT (-75.5948708 6.222133500000001),POINT (-75.5722612 6.2005726),3.463293,6.9,1,2026-01-05
2,205621,297525,297525_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,10491,2026-01-05 07:30:00-05:00,2026-01-05 07:00:00-05:00,2026-01-05 07:49:31.984874-05:00,POINT (-75.5722612 6.2005726),POINT (-75.56991479999999 6.2276503),NaN,49.5,1,2026-01-05
3,206697,298694,298694_free,FREE_TIME,FREE_TIME,31857,2026-01-05 08:00:00-05:00,2026-01-05 07:23:59.172324-05:00,2026-01-05 07:23:59.172324-05:00,POINT (-75.58290559999999 6.155980599999999),POINT (-75.58290559999999 6.155980599999999),0.000000,0.0,1,2026-01-05
4,206697,298694,298694_move,DRIVER_MOVE,DRIVER_MOVE,31857,2026-01-05 08:00:00-05:00,2026-01-05 07:23:59.172324-05:00,2026-01-05 07:30:00-05:00,POINT (-75.58290559999999 6.155980599999999),POINT (-75.5673676 6.1781754),3.006897,6.0,1,2026-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,215474,308380,308380_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-11 16:30:00-05:00,2026-01-11 15:55:44.773032-05:00,2026-01-11 16:00:00-05:00,POINT (-73.1037471 7.0910492),POINT (-73.11345413684005 7.107574257887976),2.126891,4.3,844,2026-01-11
11,215474,308380,308380_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-11 16:30:00-05:00,2026-01-11 16:00:00-05:00,2026-01-11 16:46:46.377629-05:00,POINT (-73.11345413684005 7.107574257887976),POINT (-73.1063319 7.1155142),NaN,46.8,844,2026-01-11
6,215495,308403,308403_free,FREE_TIME,FREE_TIME,6412,2026-01-11 13:00:00-05:00,2026-01-11 10:19:15.875810-05:00,2026-01-11 12:14:23.509259-05:00,POINT (-73.1134749 7.1084515),POINT (-73.1134749 7.1084515),0.000000,115.1,844,2026-01-11
7,215495,308403,308403_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-11 13:00:00-05:00,2026-01-11 12:14:23.509259-05:00,2026-01-11 12:30:00-05:00,POINT (-73.1134749 7.1084515),POINT (-73.181381 7.128081700000001),7.804090,15.6,844,2026-01-11


In [33]:
df = moves_algo_REACT_df
df = df[df['city'] != 'ALL']
df[df['service_id'] == '218944']

,service_id,labor_id,labor_context_id,labor_name,labor_category,assigned_driver,schedule_date,actual_start,actual_end,start_point,end_point,distance_km,duration_min,city,date
9,218944,312188,312188_free,FREE_TIME,FREE_TIME,6412,2026-01-05 18:00:00-05:00,2026-01-05 16:22:59.420799-05:00,2026-01-05 17:29:35.443094-05:00,POINT (-73.1149525 7.109582199999998),POINT (-73.1149525 7.109582199999998),0.000000,66.6,nan,NaN
10,218944,312188,312188_move,DRIVER_MOVE,DRIVER_MOVE,6412,2026-01-05 18:00:00-05:00,2026-01-05 17:29:35.443094-05:00,2026-01-05 17:30:00-05:00,POINT (-73.1149525 7.109582199999998),POINT (-73.11350980000002 7.108425699999999),0.204641,0.4,nan,NaN
11,218944,312188,312188_labor,Alfred Initial Transport,VEHICLE_TRANSPORTATION,6412,2026-01-05 18:00:00-05:00,2026-01-05 17:30:00-05:00,2026-01-05 18:29:36.286581-05:00,POINT (-73.11350980000002 7.108425699999999),POINT (-73.0782802 7.0281457),NaN,59.6,nan,NaN
